# Linear Regression — Predicting `duration_minutes`
### With VIF-Based Feature Selection, Ridge & Lasso Regularisation

**Goal:** Build a linear regression model (OLS + regularised variants) to predict GitHub Actions workflow `duration_minutes` using only YAML-derived and codebase features.

| Step | Description |
|------|-------------|
| 1 | Load & preprocess (type-cast, derive features) |
| 2 | IQR outlier removal on target |
| 3 | Encode categoricals (one-hot) |
| 4 | VIF-based iterative multicollinearity removal |
| 5 | Train/test split + StandardScaler |
| 6 | OLS, Ridge, Lasso regression (on log1p target) |
| 7 | Evaluation: R², Adjusted R², MAE, RMSE, MAPE |
| 8 | Residual diagnostics |
| 9 | Hyperparameter tuning tips |

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

RANDOM_STATE = 42
TARGET = 'duration_minutes'
print('Imports complete ✓')

---
## 1 · Load & Preprocess

In [ ]:
df_raw = pd.read_csv('comprehensive_features.csv')
print(f'Raw shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')
df_raw.head()

In [ ]:
# ── Drop excluded columns ──────────────────────────────────────────────
drop_cols = ['total_cost_usd', 'workflow_name', 'repo_name', 'head_sha']
df = df_raw.drop(columns=[c for c in drop_cols if c in df_raw.columns]).copy()
print(f'Shape after dropping identifiers/leaky cols: {df.shape}')

In [ ]:
# ── Type casting ──────────────────────────────────────────────────────
bool_cols = ['uses_matrix_strategy', 'is_using_setup_actions',
             'is_using_docker_actions', 'is_using_cache']
for c in bool_cols:
    if c in df.columns:
        df[c] = df[c].map({'True': 1, 'False': 0, True: 1, False: 0}).astype(int)

# fail_fast: coerce expression strings → True (default runtime value)
def parse_fail_fast(v):
    if v == 'True':  return 1
    if v == 'False': return 0
    return 1  # expression evaluates to True at runtime

if 'fail_fast' in df.columns:
    df['fail_fast'] = df['fail_fast'].apply(parse_fail_fast)

# container_image → binary flag
if 'container_image' in df.columns:
    df['has_container'] = (df['container_image'] != 'False').astype(int)
    df.drop(columns=['container_image'], inplace=True)

# yaml_depth to numeric
if 'yaml_depth' in df.columns:
    df['yaml_depth'] = pd.to_numeric(df['yaml_depth'], errors='coerce')

print('Type casting complete ✓')
df.dtypes

---
## 2 · IQR Outlier Removal on Target

In [ ]:
Q1, Q3 = df[TARGET].quantile([0.25, 0.75])
IQR = Q3 - Q1
lo, hi = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

n_before = len(df)
df = df[df[TARGET].between(lo, hi)].copy().reset_index(drop=True)
n_after = len(df)

print(f'Rows before: {n_before}')
print(f'Outliers removed: {n_before - n_after} (duration > {hi:.3f} min)')
print(f'Rows after: {n_after}')
print(f'Target range: {df[TARGET].min():.4f} – {df[TARGET].max():.4f} min')

---
## 3 · One-Hot Encode Categoricals

In [ ]:
cat_cols = ['os_label', 'primary_language']
cat_cols_present = [c for c in cat_cols if c in df.columns]

df = pd.get_dummies(df, columns=cat_cols_present, drop_first=True, dtype=int)
print(f'Shape after one-hot encoding: {df.shape}')
print(f'Columns: {list(df.columns)}')

In [ ]:
# Handle any remaining NaN values
print(f'NaN counts before fill:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
df = df.fillna(0)
print(f'\nFinal shape: {df.shape}')

---
## 4 · VIF-Based Iterative Feature Removal

Linear regression is sensitive to multicollinearity. We iteratively remove the feature with the highest VIF until all features have VIF < 10.

In [ ]:
# Separate features and target
y = df[TARGET].values
X = df.drop(columns=[TARGET])

feature_names = list(X.columns)
print(f'Starting features: {len(feature_names)}')

In [ ]:
def compute_vif(X_df):
    """Compute VIF for all features in a DataFrame."""
    X_arr = X_df.values.astype(float)
    vif_data = []
    for i in range(X_arr.shape[1]):
        vif_val = variance_inflation_factor(X_arr, i)
        vif_data.append({'Feature': X_df.columns[i], 'VIF': vif_val})
    return pd.DataFrame(vif_data).sort_values('VIF', ascending=False)

def iterative_vif_removal(X_df, threshold=10.0, verbose=True):
    """Iteratively remove the feature with highest VIF until all < threshold."""
    X_current = X_df.copy()
    dropped = []
    iteration = 0
    
    while True:
        vif_df = compute_vif(X_current)
        max_vif = vif_df['VIF'].max()
        
        if max_vif < threshold:
            if verbose:
                print(f'\n✓ All VIFs below {threshold} after {iteration} removals')
            break
        
        worst = vif_df.iloc[0]
        if verbose:
            print(f'  Iter {iteration+1}: Dropping "{worst["Feature"]}" (VIF = {worst["VIF"]:.1f})')
        dropped.append(worst['Feature'])
        X_current = X_current.drop(columns=[worst['Feature']])
        iteration += 1
    
    return X_current, dropped, vif_df

print('VIF removal function defined ✓')

In [ ]:
# ── Pre-VIF removal: show initial VIF values ─────────────────────────
print('=== Initial VIF Values ===')
vif_initial = compute_vif(X)
print(vif_initial.to_string(index=False))

In [ ]:
# ── Run iterative VIF removal ────────────────────────────────────────
print('=== Iterative VIF Removal (threshold=10) ===')
X_vif, dropped_features, vif_final = iterative_vif_removal(X, threshold=10.0)

print(f'\nDropped features: {dropped_features}')
print(f'Remaining features: {len(X_vif.columns)}')
print(f'\n=== Final VIF Values ===')
print(vif_final.to_string(index=False))

In [ ]:
# ── Visualise VIF before and after ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('VIF Analysis — Before & After Iterative Removal', fontsize=14)

# Before
ax = axes[0]
vif_before = vif_initial.sort_values('VIF', ascending=True)
colors_b = ['#ff7b72' if v > 10 else '#f0883e' if v > 5 else '#3fb950'
            for v in vif_before['VIF']]
ax.barh(vif_before['Feature'], vif_before['VIF'], color=colors_b, edgecolor='none')
ax.axvline(10, color='#ff7b72', lw=2, ls='--', label='VIF=10')
ax.axvline(5, color='#f0883e', lw=2, ls='--', label='VIF=5')
ax.set_xlabel('VIF Score')
ax.set_title('Before VIF Removal')
ax.legend()

# After
ax = axes[1]
vif_after = vif_final.sort_values('VIF', ascending=True)
colors_a = ['#ff7b72' if v > 10 else '#f0883e' if v > 5 else '#3fb950'
            for v in vif_after['VIF']]
ax.barh(vif_after['Feature'], vif_after['VIF'], color=colors_a, edgecolor='none')
ax.axvline(10, color='#ff7b72', lw=2, ls='--', label='VIF=10')
ax.axvline(5, color='#f0883e', lw=2, ls='--', label='VIF=5')
ax.set_xlabel('VIF Score')
ax.set_title('After VIF Removal')
ax.legend()

plt.tight_layout()
plt.show()

---
## 5 · Train/Test Split & Feature Scaling

In [ ]:
# Use log1p-transformed target to stabilise variance (from EDA findings)
y_log = np.log1p(y)

X_train, X_test, y_train_log, y_test_log = train_test_split(
    X_vif, y_log, test_size=0.2, random_state=RANDOM_STATE
)

# Keep original y_test for back-transformed metrics
y_test_original = np.expm1(y_test_log)
y_train_original = np.expm1(y_train_log)

# Standard scaling (important for Ridge/Lasso regularisation)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train set: {X_train_scaled.shape[0]} samples, {X_train_scaled.shape[1]} features')
print(f'Test set:  {X_test_scaled.shape[0]} samples')

---
## 6 · Model Training — OLS, Ridge, Lasso

In [ ]:
def evaluate_model(model_name, y_true_log, y_pred_log, n_features):
    """Compute metrics in both log and original scale."""
    # Log-scale metrics
    r2_log = r2_score(y_true_log, y_pred_log)
    n = len(y_true_log)
    adj_r2_log = 1 - (1 - r2_log) * (n - 1) / (n - n_features - 1)
    
    # Back-transform to original scale
    y_true_orig = np.expm1(y_true_log)
    y_pred_orig = np.expm1(y_pred_log)
    y_pred_orig = np.maximum(y_pred_orig, 0)  # clip negatives
    
    r2_orig = r2_score(y_true_orig, y_pred_orig)
    mae = mean_absolute_error(y_true_orig, y_pred_orig)
    rmse = np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))
    
    # MAPE (avoid division by zero)
    mask = y_true_orig > 0.01  # exclude near-zero durations
    if mask.sum() > 0:
        mape = np.mean(np.abs((y_true_orig[mask] - y_pred_orig[mask]) / y_true_orig[mask])) * 100
    else:
        mape = np.nan
    
    metrics = {
        'Model': model_name,
        'R² (log)': round(r2_log, 4),
        'Adj R² (log)': round(adj_r2_log, 4),
        'R² (original)': round(r2_orig, 4),
        'MAE (min)': round(mae, 4),
        'RMSE (min)': round(rmse, 4),
        'MAPE (%)': round(mape, 2),
    }
    return metrics

print('Evaluation function defined ✓')

In [ ]:
# ── 6a. Ordinary Least Squares ──────────────────────────────────────
ols = LinearRegression()
ols.fit(X_train_scaled, y_train_log)
y_pred_ols = ols.predict(X_test_scaled)

metrics_ols = evaluate_model('OLS', y_test_log, y_pred_ols, X_train_scaled.shape[1])
print('OLS Regression Results:')
for k, v in metrics_ols.items():
    print(f'  {k}: {v}')

In [ ]:
# ── 6b. Ridge Regression (L2) with Cross-Validated alpha ─────────────
alphas = np.logspace(-4, 4, 100)
ridge_cv = RidgeCV(alphas=alphas, scoring='r2', cv=5)
ridge_cv.fit(X_train_scaled, y_train_log)

print(f'Best Ridge alpha: {ridge_cv.alpha_:.6f}')

ridge = Ridge(alpha=ridge_cv.alpha_)
ridge.fit(X_train_scaled, y_train_log)
y_pred_ridge = ridge.predict(X_test_scaled)

metrics_ridge = evaluate_model('Ridge', y_test_log, y_pred_ridge, X_train_scaled.shape[1])
print('\nRidge Regression Results:')
for k, v in metrics_ridge.items():
    print(f'  {k}: {v}')

In [ ]:
# ── 6c. Lasso Regression (L1) with Cross-Validated alpha ─────────────
lasso_cv = LassoCV(alphas=alphas, cv=5, random_state=RANDOM_STATE, max_iter=10000)
lasso_cv.fit(X_train_scaled, y_train_log)

print(f'Best Lasso alpha: {lasso_cv.alpha_:.6f}')

lasso = Lasso(alpha=lasso_cv.alpha_, max_iter=10000)
lasso.fit(X_train_scaled, y_train_log)
y_pred_lasso = lasso.predict(X_test_scaled)

# Show which features Lasso zeroed out
lasso_coefs = pd.Series(lasso.coef_, index=X_vif.columns)
zeroed = lasso_coefs[lasso_coefs == 0].index.tolist()
print(f'Features zeroed out by Lasso: {zeroed if zeroed else "None"}')

metrics_lasso = evaluate_model('Lasso', y_test_log, y_pred_lasso, X_train_scaled.shape[1])
print('\nLasso Regression Results:')
for k, v in metrics_lasso.items():
    print(f'  {k}: {v}')

In [ ]:
# ── 6d. Statsmodels OLS Summary (for p-values & diagnostics) ─────────
X_train_sm = sm.add_constant(X_train_scaled)
ols_sm = sm.OLS(y_train_log, X_train_sm).fit()
print(ols_sm.summary())

---
## 7 · Model Comparison & Cross-Validation

In [ ]:
# ── Summary table ────────────────────────────────────────────────────
results_df = pd.DataFrame([metrics_ols, metrics_ridge, metrics_lasso])
print('=== Model Comparison ===')
display(results_df)

In [ ]:
# ── Cross-validation for robustness ──────────────────────────────────
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

X_all_scaled = scaler.fit_transform(X_vif)

models = {
    'OLS': LinearRegression(),
    'Ridge': Ridge(alpha=ridge_cv.alpha_),
    'Lasso': Lasso(alpha=lasso_cv.alpha_, max_iter=10000),
}

print('=== 5-Fold Cross-Validation R² (on log1p target) ===')
cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_all_scaled, y_log, cv=kf, scoring='r2')
    cv_results[name] = scores
    print(f'  {name:8s}: mean R² = {scores.mean():.4f} ± {scores.std():.4f}  '
          f'[{scores.min():.4f}, {scores.max():.4f}]')

In [ ]:
# ── CV boxplot ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([cv_results[m] for m in models],
           labels=list(models.keys()), patch_artist=True,
           boxprops=dict(facecolor='#58a6ff', alpha=0.7),
           medianprops=dict(color='white', lw=2))
ax.set_ylabel('R² Score (log1p target)')
ax.set_title('5-Fold CV Performance Comparison')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 8 · Feature Coefficients Analysis

In [ ]:
# ── Coefficient comparison across models ─────────────────────────────
coef_df = pd.DataFrame({
    'Feature': X_vif.columns,
    'OLS': ols.coef_,
    'Ridge': ridge.coef_,
    'Lasso': lasso.coef_,
})
coef_df['|OLS|'] = coef_df['OLS'].abs()
coef_df = coef_df.sort_values('|OLS|', ascending=False)

fig, ax = plt.subplots(figsize=(14, 7))
x_idx = np.arange(len(coef_df))
w = 0.25
ax.bar(x_idx - w, coef_df['OLS'],   w, label='OLS',   color='#58a6ff', alpha=0.85)
ax.bar(x_idx,     coef_df['Ridge'], w, label='Ridge', color='#f0883e', alpha=0.85)
ax.bar(x_idx + w, coef_df['Lasso'], w, label='Lasso', color='#3fb950', alpha=0.85)
ax.axhline(0, color='white', lw=0.8)
ax.set_xticks(x_idx)
ax.set_xticklabels(coef_df['Feature'], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Standardised Coefficient')
ax.set_title('Feature Coefficients — OLS vs Ridge vs Lasso (sorted by |OLS|)')
ax.legend()
plt.tight_layout()
plt.show()

---
## 9 · Residual Diagnostics

In [ ]:
# Use the best model (Ridge) for residual analysis
best_model_name = results_df.loc[results_df['R² (log)'].idxmax(), 'Model']
best_preds = {'OLS': y_pred_ols, 'Ridge': y_pred_ridge, 'Lasso': y_pred_lasso}
y_pred_best = best_preds[best_model_name]

residuals_log = y_test_log - y_pred_best
y_pred_orig = np.expm1(y_pred_best)
residuals_orig = y_test_original - y_pred_orig

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Residual Diagnostics — {best_model_name} Regression', fontsize=14)

# 1. Residuals vs Fitted (log scale)
ax = axes[0, 0]
ax.scatter(y_pred_best, residuals_log, alpha=0.3, s=12, color='#58a6ff')
ax.axhline(0, color='white', lw=1, ls='--')
ax.set_xlabel('Fitted values (log scale)')
ax.set_ylabel('Residuals (log scale)')
ax.set_title('Residuals vs Fitted')

# 2. Q-Q plot of residuals
ax = axes[0, 1]
(osm, osr), (slope, intercept, r_qq) = stats.probplot(residuals_log, dist='norm')
ax.scatter(osm, osr, color='#58a6ff', s=10, alpha=0.5)
xl = np.linspace(min(osm), max(osm), 100)
ax.plot(xl, slope * xl + intercept, color='#ff7b72', lw=2)
ax.set_xlabel('Theoretical Quantiles')
ax.set_ylabel('Sample Quantiles')
ax.set_title(f'Q-Q Plot (r={r_qq:.3f})')

# 3. Histogram of residuals
ax = axes[1, 0]
ax.hist(residuals_log, bins=50, color='#58a6ff', alpha=0.7, edgecolor='none', density=True)
x_range = np.linspace(residuals_log.min(), residuals_log.max(), 200)
ax.plot(x_range, stats.norm.pdf(x_range, residuals_log.mean(), residuals_log.std()),
        color='#ff7b72', lw=2, label='Normal fit')
ax.set_xlabel('Residuals (log scale)')
ax.set_ylabel('Density')
ax.set_title('Residual Distribution')
ax.legend()

# 4. Actual vs Predicted (original scale)
ax = axes[1, 1]
ax.scatter(y_test_original, y_pred_orig, alpha=0.3, s=12, color='#3fb950')
lim = max(y_test_original.max(), y_pred_orig.max())
ax.plot([0, lim], [0, lim], 'w--', lw=1.5, alpha=0.6, label='Perfect prediction')
ax.set_xlabel('Actual duration (min)')
ax.set_ylabel('Predicted duration (min)')
ax.set_title('Actual vs Predicted (original scale)')
ax.legend()

plt.tight_layout()
plt.show()

---
## 10 · Hyperparameter Tuning Tips for Linear Regression

### OLS (Ordinary Least Squares)
- **No hyperparameters** to tune — OLS is a closed-form solution.
- Focus on **feature engineering** and **feature selection** (VIF, Lasso zero-out).
- Consider polynomial features for non-linear relationships (e.g., `PolynomialFeatures(degree=2, interaction_only=True)`).

### Ridge Regression (L2 Regularisation)
- **`alpha`** — Regularisation strength. Higher α → more shrinkage → less overfitting.
  - Use `RidgeCV` with `alphas=np.logspace(-4, 4, 100)` for automatic selection.
  - Monitor the trade-off: too high α underestimates important coefficients.
- **When to use:** When multicollinearity remains even after VIF removal, or when you have many features relative to samples.

### Lasso Regression (L1 Regularisation)
- **`alpha`** — Controls sparsity. Higher α → more coefficients pushed to exactly 0.
  - Use `LassoCV` with `cv=5` and a log-spaced alpha grid.
  - Inspect which features survive (non-zero coefficients) — this is **automatic feature selection**.
- **`max_iter`** — Increase to 10000+ if convergence warnings appear.
- **When to use:** When you suspect many features are irrelevant and want automatic pruning.

### ElasticNet (L1 + L2)
- Combines Ridge and Lasso via `l1_ratio` (0 = pure Ridge, 1 = pure Lasso).
- Use `ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0])` to find the best blend.

### General Tips
- **Always scale features** before regularised regression (StandardScaler).
- **log1p transform** the target if it's right-skewed (as in this dataset).
- Try **interaction features** between strong predictors (e.g., `total_steps × os_label`).
- Use **VIF < 5** (stricter threshold) for purely interpretive models.
- **Adjusted R²** is more reliable than R² when comparing models with different numbers of features.

In [ ]:
print('═══ LINEAR REGRESSION SUMMARY ═══════════════════════════════════════════')
print()
print(f'Best model: {best_model_name}')
best_metrics = results_df[results_df['Model'] == best_model_name].iloc[0]
print(f'  R² (log scale):      {best_metrics["R² (log)"]}')
print(f'  Adj R² (log scale):  {best_metrics["Adj R² (log)"]}')
print(f'  R² (original scale): {best_metrics["R² (original)"]}')
print(f'  MAE:                 {best_metrics["MAE (min)"]} min')
print(f'  RMSE:                {best_metrics["RMSE (min)"]} min')
print(f'  MAPE:                {best_metrics["MAPE (%)"]}%')
print()
print(f'Features used ({len(X_vif.columns)}):')
for f in X_vif.columns:
    print(f'  • {f}')
print()
if dropped_features:
    print(f'Features dropped by VIF ({len(dropped_features)}):')
    for f in dropped_features:
        print(f'  ✗ {f}')
print()
if zeroed:
    print(f'Features zeroed by Lasso: {zeroed}')
print('\n✓ Linear Regression notebook complete.')